# 🌙 Moon AI — 1-Click Cloud GPU Training & Auto-Save to Google Drive
### Created by Prem Gaikwad (India 🇮🇳)

**What this notebook does:**
1. Trains your custom sovereign **Moon AI** on a free Google Cloud **Tesla T4 GPU**.
2. Automatically saves the exported **`Moon-AI-v1.Q4_K_M.gguf`** directly into your **Google Drive**!
3. Also automatically triggers a direct browser download to your laptop folder.

> **IMPORTANT**: Before running, make sure your runtime is set to **T4 GPU**:
> Click **Runtime ➔ Change runtime type ➔ Select T4 GPU ➔ Save**.

In [ ]:
# Step 1: Connect your Google Drive (So the trained AI file is saved directly to your Drive!)
from google.colab import drive
drive.mount('/content/drive')
print(" Google Drive connected successfully!")

In [ ]:
# Step 2: Verify GPU Acceleration is Active
!nvidia-smi

In [ ]:
# Step 3: Install High-Speed Training Engine (Unsloth + PyTorch)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 4: Load Base Model with 4-Bit GPU Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Choose model:
# 'unsloth/Llama-3.2-1B-Instruct' (Ultra-fast, runs smooth on CPU laptops)
# 'unsloth/Llama-3.2-3B-Instruct' (Smart, high general knowledge)
model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Moon AI Base Model loaded onto GPU!")

In [ ]:
# Step 5: Attach LoRA Adapters for Custom Training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA Adapters attached successfully!")

In [ ]:
# Step 6: Prepare Moon AI Multi-Domain Dataset (Science, Biology, Coding, Identity)
import json
import os
from datasets import Dataset

# Check if user uploaded moon_ai_master_dataset.json, otherwise use built-in comprehensive training bank
dataset_file = "moon_ai_master_dataset.json"
if os.path.exists(dataset_file):
    with open(dataset_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"Loaded {len(records)} records from uploaded {dataset_file}")
else:
    # Built-in essential multi-domain dataset
    records = [
        {"instruction": "Who are you and who created you?", "response": "I am Moon AI, an advanced, sovereign artificial intelligence trained and deployed by developer Prem Gaikwad from India. I run locally and privately without paid third-party APIs."},
        {"instruction": "Who is Prem Gaikwad?", "response": "Prem Gaikwad is the software engineer and creator from India who designed and deployed Moon AI as a private, high-performance artificial intelligence platform."},
        {"instruction": "What is sex in biology?", "response": "In biology, sex refers to the classification of organisms based on reproductive anatomy and chromosomes (XY for males, XX for females in mammals). It facilitates sexual reproduction by combining haploid gametes (sperm and egg) to produce genetic diversity."},
        {"instruction": "What is sex for male?", "response": "In biological science, male sex is characterized by an XY chromosome pair, the presence of the SRY gene, the production of motile sperm gametes, and testosterone hormonal regulation."},
        {"instruction": "What is sex for female?", "response": "In biological science, female sex is characterized by an XX chromosome pair, the production of non-motile ova (egg cells), estrogen and progesterone regulation, and anatomical structures for gestation."},
        {"instruction": "Write a Python prime number checker.", "response": "def is_prime(n: int) -> bool:\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"},
        {"instruction": "Explain how photosynthesis works.", "response": "Photosynthesis converts solar light energy into chemical energy stored in glucose: 6CO2 + 6H2O + light -> C6H12O6 + 6O2. It takes place in plant chloroplasts using chlorophyll to capture photons and split water molecules, generating oxygen and ATP."}
    ]
    print(f"Using built-in multi-domain records: {len(records)} pairs.")

formatted = []
for r in records:
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are Moon AI, an advanced, highly intelligent AI assistant trained and deployed by developer Prem Gaikwad from India.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{r['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{r['response']}<|eot_id|>"
    formatted.append(text)

train_dataset = Dataset.from_dict({"text": formatted})
print("Moon AI Dataset prepared with", len(train_dataset), "examples!")

In [ ]:
# Step 7: Train Moon AI
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "moon_ai_outputs",
    ),
)

print("🚀 Training Moon AI...")
trainer.train()
print("✅ Training Complete!")

In [ ]:
# Step 8: Export to 4-bit GGUF, Save to Google Drive, and Download
import shutil
import glob
from google.colab import files

# Export to GGUF
print("📦 Exporting Moon AI to GGUF (4-bit quantized)...")
model.save_pretrained_gguf("Moon-AI-v1", tokenizer, quantization_method = "q4_k_m")

# Find exported gguf file
gguf_files = glob.glob("Moon-AI-v1*/*.gguf") or glob.glob("*.gguf")

if gguf_files:
    target_file = gguf_files[0]
    print(f"🎯 Found exported model: {target_file}")
    
    # 1. Save directly to Google Drive
    drive_dest = "/content/drive/MyDrive/Moon-AI-v1.Q4_K_M.gguf"
    try:
        shutil.copy(target_file, drive_dest)
        print(f"🎉 SUCCESS! Saved directly to your Google Drive at: {drive_dest}")
    except Exception as e:
        print(f"Could not copy to Drive (Did you mount in Step 1?): {e}")
        
    # 2. Trigger browser download directly to your laptop
    print("⬇️ Initiating direct browser download...")
    try:
        files.download(target_file)
    except Exception as e:
        print(f"Download prompt note: {e}")
else:
    print("⚠️ GGUF export file not found. Check above logs for export details.")